In [12]:
import os
import kaggle
import pandas as pd

# Download dataset from Kaggle
def download_dataset():
    try:
        # Download the dataset
        !kaggle datasets download -d sujithmandala/credit-score-classification-dataset
        
        # Unzip the downloaded file
        !unzip -o credit-score-classification-dataset.zip
        
        print("Dataset downloaded successfully")
        return True
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return False

# Run this before your main Flask application
download_dataset()

Dataset URL: https://www.kaggle.com/datasets/sujithmandala/credit-score-classification-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
credit-score-classification-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  credit-score-classification-dataset.zip
  inflating: Credit Score Classification Dataset.csv  
Dataset downloaded successfully


True

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'apikeyhere'  

In [14]:
data = pd.read_csv('/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv')


In [15]:
import os
import psutil
import socket

def find_process_using_port(port):
    """Find and terminate a process using the given port."""
    for conn in psutil.net_connections(kind='inet'):
        if conn.laddr.port == port:
            pid = conn.pid
            if pid:
                process = psutil.Process(pid)
                print(f"Port {port} is in use by process: {process.name()} (PID {pid})")
                return process
    return None

In [16]:
def free_port(port):
    """Kill the process using the given port."""
    process = find_process_using_port(port)
    if process:
        process.terminate()
        print(f"Terminated process {process.name()} (PID {process.pid}) using port {port}.")
    else:
        print(f"No process found using port {port}.")

def is_port_available(port):
    """Check if a port is available."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) != 0

In [17]:
port = 5001  
if not is_port_available(port):
    print(f"Port {port} is in use. Attempting to free it...")
    free_port(port)
else:
    print(f"Port {port} is available.")

Port 5001 is available.


In [19]:
import os
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


# Load environment variables
load_dotenv()

class APIGenerator:
    def __init__(self, api_key: str):
        """Initialize the API Generator with OpenAI API key."""
        self.client = OpenAI(api_key=api_key)

    def generate_api_code(self, prompt: str) -> str:
        try:
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "Generate a complete, syntactically correct Python FastAPI application. Do not include triple backticks, language specifiers, or explanatory comments at the end."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=2000
            )
            
            if not response.choices:
                raise ValueError("No response received from OpenAI API.")
            
            generated_code = response.choices[0].message.content.strip()
            
            # Remove triple backticks if present
            generated_code = generated_code.replace('``````', '').strip()
            
            # Remove any trailing comments or explanations
            generated_code = generated_code.split('\n')
            while generated_code and (generated_code[-1].strip().startswith('#') or not generated_code[-1].strip()):
                generated_code.pop()
            generated_code = '\n'.join(generated_code)
            
            print("\nGenerated API Code:")
            print("=" * 50)
            print(generated_code)
            print("=" * 50)
            
            return generated_code
        except Exception as e:
            print(f"Error generating API code: {e}")
            return ""

    def save_and_run_code(self, code: str) -> None:
        try:
            # Save the generated code to a file in the current directory
            file_name = "generated_api.py"
            with open(file_name, "w") as f:
                f.write(code)

            print(f"\nSaved generated code to: {os.path.abspath(file_name)}")

            # Run the generated code
            print("\nRunning the generated FastAPI application...")
            subprocess.run(["uvicorn", f"{file_name[:-3]}:app", "--reload"], check=True)

        except subprocess.CalledProcessError as e:
            print(f"Error running the FastAPI application: {e}")
        except Exception as e:
            print(f"Error saving or running generated API code: {e}")

if __name__ == "__main__":
    api_key = os.getenv('OPENAI_API_KEY')

    if not api_key:
        print("Error: Please set the OPENAI_API_KEY environment variable.")
        sys.exit(1)

    api_generator = APIGenerator(api_key)

    prompt = """
    Create a FastAPI application for a loan approval system with the following specifications:

    1. Use the dataset located at '/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv'.

    2. Implement the following endpoints:
       a. POST /predict: Takes user details and predicts loan approval
       b. GET /status/{user_id}: Fetch loan status for a specific user
       c. GET /all: Get all loan approvals

    3. Data preprocessing:
       - Handle categorical variables using LabelEncoder
       - Split the data into training and testing sets

    4. Use RandomForestClassifier for the prediction model

    5. Implement proper error handling for each endpoint

    6. Use Pydantic models for request/response validation

    7. Ensure the code is properly formatted and follows Python best practices

    8. Include necessary imports at the beginning of the file

    9. Add comments to explain key parts of the code

    10. Make sure all string literals are properly terminated

    11. Use FastAPI's built-in exception handling

    The dataset columns are: ['Age', 'Gender', 'Income', 'Education', 'Marital Status', 'Number of Children', 'Home Ownership', 'Credit Score']

    Do not include any explanatory comments at the end of the code.
    The error you're encountering is due to extra text being included at the beginning of the generated code file. The FastAPI application code should start directly with the Python imports, not with explanatory text.

example:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Define Pydantic models
class User(BaseModel):
    Age: int
    Gender: str
    Income: int
    Education: str
    Marital_Status: str = Field(..., alias='Marital Status')
    Number_of_Children: int
    Home_Ownership: str = Field(..., alias='Home Ownership')
    Credit_Score: int

class LoanStatus(BaseModel):
    user_id: int
    approval_status: str

# Initialize FastAPI
app = FastAPI()

# Load the dataset
df = pd.read_csv('/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv')

# Preprocessing
encoder = LabelEncoder()
categorical_features = ['Gender', 'Education', 'Marital Status', 'Home Ownership']
df[categorical_features] = df[categorical_features].apply(encoder.fit_transform)

# Split the data
X = df.drop('Credit Score', axis=1)  # Assuming 'Credit Score' is the target variable
y = df['Credit Score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Define the endpoints
@app.post("/predict")
async def predict_loan_approval(user: User):
    try:
        input_data = pd.DataFrame([user.dict(by_alias=True)])
        input_data = input_data.rename(columns={
            'Marital_Status': 'Marital Status',
            'Home_Ownership': 'Home Ownership'
        })
        prediction = model.predict(input_data)[0]
        return {'Loan approval prediction': 'Approved' if prediction == 1 else 'Rejected'}
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error in prediction: {str(e)}")

@app.get("/status/{user_id}", response_model=LoanStatus)
async def get_loan_status(user_id: int):
    try:
        status = df.loc[df.index == user_id, 'Credit Score'].values[0]
        return {'user_id': user_id, 'approval_status': 'Approved' if status == 1 else 'Rejected'}
    except IndexError:
        raise HTTPException(status_code=404, detail="User not found")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error fetching status: {str(e)}")

@app.get("/all", response_model=List[LoanStatus])
async def get_all_loan_approvals():
    try:
        all_status = [{'user_id': index, 'approval_status': 'Approved' if score == 1 else 'Rejected'} 
                      for index, score in df['Credit Score'].items()]
        return all_status
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error fetching all loan approvals: {str(e)}")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


    """

    print("Generating and running API...")
    generated_code = api_generator.generate_api_code(prompt)
    if generated_code:
        api_generator.save_and_run_code(generated_code)
    else:
        print("Failed to generate API code.")


Generating and running API...

Generated API Code:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

class User(BaseModel):
    Age: int
    Gender: str
    Income: int
    Education: str
    Marital_Status: str = Field(..., alias='Marital Status')
    Number_of_Children: int = Field(..., alias='Number of Children')
    Home_Ownership: str = Field(..., alias='Home Ownership')
    Credit_Score: int = Field(..., alias='Credit Score')

class LoanStatus(BaseModel):
    user_id: int
    approval_status: str

app = FastAPI()

df = pd.read_csv('/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv')

encoder = LabelEncoder()
categorical_features = ['Gender', 'Education', 'Marital Status', 'Home Ownership']
df[categorical_features] = df[categorical

INFO:     Will watch for changes in these directories: ['/Users/kavinraj/banking_agents/api_gen']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [24005] using StatReload
INFO:     Started server process [24007]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [24007]
INFO:     Stopping reloader process [24005]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# Load dataset
file_path = "/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv"
df = pd.read_csv(file_path)

# Rename target column
df.rename(columns={'Credit Score': 'Loan_Status'}, inplace=True)

# Convert categorical features to numeric
categorical_columns = ['Gender', 'Education', 'Marital Status', 'Home Ownership']
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)  # One-hot encoding

# Encode target variable ('Loan_Status')
label_encoder = LabelEncoder()
df['Loan_Status'] = label_encoder.fit_transform(df['Loan_Status'])  # Convert to 0,1,2

# Define features and target
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Train classifier
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

# Save model
joblib.dump(classifier, "model.pkl")

print("Model saved as model.pkl")


Model saved as model.pkl


In [ ]:
import pandas as pd

# Load the dataset
data = pd.read_csv('/Users/kavinraj/Downloads/Credit Score Classification Dataset.csv')

# Print the column names
print(data.columns.tolist())


['Age', 'Gender', 'Income', 'Education', 'Marital Status', 'Number of Children', 'Home Ownership', 'Credit Score']


In [ ]:
import sqlite3
import datetime

class CodeStorage:
    def __init__(self, db_name='generated_codes.db'):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
        self.create_table()

    def create_table(self):
        self.cursor.execute('''
        CREATE TABLE IF NOT EXISTS codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            code TEXT NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
        ''')
        self.conn.commit()

    def store_code(self, code):
        self.cursor.execute("INSERT INTO codes (code) VALUES (?)", (code,))
        self.conn.commit()

    def get_all_codes(self):
        self.cursor.execute("SELECT * FROM codes")
        return self.cursor.fetchall()

    def close(self):
        self.conn.close()

# Usage in your existing code:
code_storage = CodeStorage()

# After generating code:
generated_code = api_generator.generate_api_code(prompt)
if generated_code:
    code_storage.store_code(generated_code)
    api_generator.save_and_run_code(generated_code)

# When you're done:
code_storage.close()



Generated API Code:
Your code appears to be correct and complete. It includes all necessary imports and follows the specifications given. It also includes proper error handling with FastAPI's built-in exception handling. The code is also well-formatted and follows Python best practices. Good job!

Saved generated code to: /Users/kavinraj/banking_agents/api_gen/generated_api.py

Running the generated FastAPI application...


INFO:     Will watch for changes in these directories: ['/Users/kavinraj/banking_agents/api_gen']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [23260] using StatReload
Process SpawnProcess-1:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.12/site-packages/uvicorn/_subprocess.py", line 80, in subprocess_started
    target(sockets=sockets)
  File "/opt/anaconda3/lib/python3.12/site-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/asyncio/runners.py", line 194, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/opt/

KeyboardInterrupt: 